# Melanoma Detection

In [1]:
# Imports
import os
import shutil
import random
import kagglehub
import polars as pl
import torch
import torchvision

/home/cam/miniforge3/envs/jupyter_dl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Preparing Data

In [2]:
# Download latest version
path = kagglehub.dataset_download("andrewmvd/isic-2019")

print("Path to dataset files:", path)

100%|██████████████████████████████████████| 9.10G/9.10G [02:34<00:00, 63.2MB/s]

Extracting files...


Path to dataset files: /home/cam/.cache/kagglehub/datasets/andrewmvd/isic-2019/versions/1


In [3]:
!ls {path}

ISIC_2019_Training_GroundTruth.csv  ISIC_2019_Training_Metadata.csv
ISIC_2019_Training_Input


In [4]:
df_train_meta = pl.read_csv(os.path.join(path, "ISIC_2019_Training_Metadata.csv"))
df_train_labels = pl.read_csv(os.path.join(path, "ISIC_2019_Training_GroundTruth.csv"))
df_train_meta.head()

image,age_approx,anatom_site_general,lesion_id,sex
str,i64,str,str,str
"""ISIC_0000000""",55,"""anterior torso""",null,"""female"""
"""ISIC_0000001""",30,"""anterior torso""",null,"""female"""
"""ISIC_0000002""",60,"""upper extremity""",null,"""female"""
"""ISIC_0000003""",30,"""upper extremity""",null,"""male"""
"""ISIC_0000004""",80,"""posterior torso""",null,"""male"""


In [5]:
df_train_labels.head()

image,MEL,NV,BCC,AK,BKL,DF,VASC,SCC,UNK
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ISIC_0000000""",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""ISIC_0000001""",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""ISIC_0000002""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""ISIC_0000003""",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""ISIC_0000004""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
df_train_labels.filter(pl.col("MEL") == 1)["image"]

image
str
"""ISIC_0000002"""
"""ISIC_0000004"""
"""ISIC_0000013"""
"""ISIC_0000022_downsampled"""
"""ISIC_0000026_downsampled"""
…
"""ISIC_0073231"""
"""ISIC_0073237"""
"""ISIC_0073238"""


In [7]:
df_train_labels.shape[0]

25331

In [8]:
# Define train test split
train_size = 0.9
test_size = 0.1

train_len = int(df_train_labels.shape[0] * 0.9)
test_len = df_train_labels.shape[0] - train_len
is_train = ([True] * train_len) + ([False] * test_len)
random.shuffle(is_train)
is_train[:10], len(is_train)

([True, True, True, True, True, True, True, True, False, True], 25331)

In [9]:
# Move data into folders for Pytorch Data Loader
image_path = os.path.join(path, "ISIC_2019_Training_Input", "ISIC_2019_Training_Input")
train_path = os.path.join(image_path, "train")
test_path = os.path.join(image_path, "test")
os.makedirs(train_path, exist_ok=True)
os.makedirs(test_path, exist_ok=True)

idx = 0
for label in df_train_labels.columns[1:]:
    # make a directory for each label
    os.makedirs(os.path.join(train_path, label), exist_ok=True)
    os.makedirs(os.path.join(test_path, label), exist_ok=True)
    
    # move the corresponding images to that directory
    for image_name in df_train_labels.filter(pl.col(label) == 1)["image"]:
        folder = train_path if is_train[idx] else test_path
        image = os.path.join(image_path, f"{image_name}.jpg")
        idx += 1 
        
        if os.path.exists(image):
            shutil.copy(image, os.path.join(folder, label, f"{image_name}.jpg"))
        

In [11]:
# PROBLEM: Some labels have no samples. Figure out a way to get uniform sampling across classes

In [10]:
train_ds = torchvision.datasets.ImageFolder(train_path)

FileNotFoundError: Found no valid file for the classes UNK. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp